# Table of DataFrames

**DataFrame Name** | **Description** | **Based on** | **Exported as**
------- | --------------- | ------------ | -------------
df_orig_k | raw DataFrame for 'Kraftwerksliste' | Kraftwerksliste_CSV.txt |
df_orig_p | raw DataFrame for 'electricity spot market price' | energy_charts_electricity_spot_market_price_germany_from2015.csv | df_monthly_electricity_price.csv
 df_orig_g | raw DataFrame for Germany's solar generation capacity | energy_charts_generation_capacity_germany_from2015.csv | df_monthly_solar_capacity.csv
df_orig_w | raw DataFrame for Germany's monthly weather features | weather_monthly_de_from2015.csv | df_monthly_weather_average.csv

# Set up EDA environment

In [ ]:
# %pip install missingno

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.dates import MonthLocator
import seaborn as sns
import pytz
from timeit import default_timer as timer
import pprint
import missingno as msno

import statsmodels.formula.api as smf
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.ndimage import gaussian_filter
import calendar
from calendar import monthrange
from calendar import month_name

from sklearn.metrics import mean_squared_error
from math import sqrt
from statsmodels.tsa.stattools import adfuller,kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima_model import ARIMA
from statsmodels.graphics.tsaplots import plot_pacf

from pmdarima.arima import auto_arima
import statsmodels.graphics.tsaplots as tsaplot
from statsmodels.tsa.holtwinters import Holt, ExponentialSmoothing, SimpleExpSmoothing

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.options.display.max_rows = 1000
# pd.options.display.width = 100000

# Set plotting style
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.figsize'] = (6, 4)
# Candidate styles: ggplot, fivethirtyeight, solarized, seaborn-paper, paired, set1
# plt.style.use('seaborn-paper')
# style for Seaborn must be one of white, dark, whitegrid, darkgrid, ticks
# sns.set_style('whitegrid')

%matplotlib inline
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

In [ ]:
# Create a new directory for all the outputs

path = '../output'

# Check if the specified path exists or not
is_exist = os.path.exists(path)
if not is_exist:
   os.makedirs(path)    # Create a new directory if it does not already exist

In [ ]:
# Define the 'info_describe' function

from IPython.display import display

def info_describe(df, N_MOST_FREQ=5):
    
    # Display top-N_MOST_FREQ most frequent unique values

    # Get basic statistics for numeric columns using df.describe()
    describe_numeric = df.describe().transpose()

    # Combine the information into a single DataFrame
    feature_info_list = []

    for i, col in enumerate(df.columns):
        
        data_type = df[col].dtype
        non_null_count = df[col].count()
        unique_count = df[col].nunique()
        top = df[col].mode().iloc[0] if non_null_count > 0 else None
        frequency = df[col].value_counts().to_dict() if non_null_count > 0 else None
        if data_type != 'object' and data_type != 'datetime64[ns]' and data_type != 'datetime64[ns, pytz.FixedOffset(60)]':  # Numeric columns
            describe_numeric_values = describe_numeric.loc[col].tolist()
        else:
            describe_numeric_values = [None] * 8  # Placeholder values for non-numeric columns (number of columns from output of df.describe().transpose())

        if frequency is not None and len(frequency) > N_MOST_FREQ:  # Limit to top-N_MOST_FREQ most frequent unique values
            top_n_freq = dict(list(frequency.items())[:N_MOST_FREQ])
            frequency_str = str(top_n_freq)[:-1] + ', ...}'
        else:
            frequency_str = str(frequency)

        feature_info = {
            # 'Feature Index': i,
            'Feature Name': col,
            'Data Type': data_type,
            'Non-Null Count': non_null_count,
            'Num of Unique Values': unique_count,
            'Mean': describe_numeric_values[1],
            'Std': describe_numeric_values[2],
            'Min': describe_numeric_values[3],
            '25%': describe_numeric_values[4],
            '50%': describe_numeric_values[5],
            '75%': describe_numeric_values[6],
            'Max': describe_numeric_values[7],
            'Most Frequent Value': top,
            'Most Frequent Unique Values': frequency_str
        }
        feature_info_list.append(feature_info)

    # Create the final DataFrame
    feature_info_df = pd.DataFrame(feature_info_list)

    # Print the final DataFrame using display()
    print("\nDataFrame Description and Unique Value Counts:")

    # No wrapping for the display of the final DataFrame
    feature_info_df_nowrap = feature_info_df.style.set_table_styles([dict(selector="td", props=[('white-space', 'nowrap')])])
    display(feature_info_df_nowrap)

# "Kraftwerksliste"

## - Import and inspect data

In [ ]:
file_path = '../data/monthly_features/Kraftwerksliste_CSV.txt'

# Read the CSV file with 'ISO-8859-1' encoding, skip the first 10 rows, and use the 11th row as the header
df_orig_k = pd.read_csv(file_path, delimiter=';', skiprows=10, encoding='ISO-8859-1')

df_orig_k.head()


In [ ]:
df = df_orig_k.copy()
print(df.shape)
df.info()

In [ ]:
info_describe(df, 2)

In [ ]:
cols_german = df_orig_k.columns
cols_english = ['mastr_nr', 'plant_operator',
       'plant_name', 'zipcode',
       'city', 'street_name', 'street_nr',
       'federal_state',
       'date_first_commission',
       'date_final_decommission',
       'plant_status', 'energy_source', 'main_fuel',
       'analysis_energy_source', 'heat_extraction',
       'is_part_of_marginal_plant',
       'gross_output_power_mw',
       'net_output_power_mw',
       'marginal_plant_net_output_power_de_mw',
       'electricity_generation_technology',
       'full_or_partial_feed_in', 'network_operator',
       'voltage_level']
df.columns = cols_english

cols_translation = {}
num_cols = len(cols_german)
for i in range(num_cols):
    cols_translation[cols_german[i]] = cols_english[i]

print("Translation for column names:")
# print("\n".join("{}\t{}".format(k, v) for k, v in cols_translation.items()))
for k, v in cols_translation.items():
    print(f"{k:>100} : {v:<50}")


In [ ]:
df['full_street_address'] = df['street_name'] + ' ' + df['street_nr']
df.head(2)

In [ ]:
# Drop duplicates and reset indices

duplicates = df.duplicated()
print(df[duplicates])

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

duplicates = df.duplicated()
print(df[duplicates])

In [ ]:
# Overview of missing values

msno.matrix(df, labels=True)

In [ ]:
print(df['date_final_decommission'].value_counts())
print(df['federal_state'].value_counts())
print(df['plant_status'].value_counts())
print(df['energy_source'].value_counts())
print(df['main_fuel'].value_counts())

In [ ]:
df_solar = df.query("energy_source == 'Solare Strahlungsenergie'")
df_solar.head(100)
# df_solar['date_first_commission']

!! Solar power plants in "Kraftwerksliste" are too few and have no commission date. Aborted.

# Monthly electricity price, solar generation capacity and weather data

# Monthly electricity price

## - Import and inspect data

In [ ]:
file_path = '../data/monthly_features/energy_charts_electricity_spot_market_price_germany_from2015.csv'

df_orig_p = pd.read_csv(file_path, delimiter=',')

df_orig_p.head()

In [ ]:
df = df_orig_p.copy()
print(df.shape)
df.info()

In [ ]:
df.head(120)

In [ ]:
info_describe(df, 2)

In [ ]:
print(df.columns)

In [ ]:
cols_orig = df_orig_p.columns
cols_py = ['month_dot_year', 'spot_market_price_eur_per_mwh',
       'day_ahead_auction_volume_weighted_eur_per_mwh',
       'intraday_continuous_volume_weighted_eur_per_mwh',
       'intraday_continuous_30min_volume_weighted_eur_per_mwh',
       'intraday_continuous_15min_volume_weighted_eur_per_mwh',
       'intraday_auction_15_minutes_call_eur_per_mwh',
       'co2_emission_allowances_auction_eur_per_tco2', 'gas_ncg_eur_per_tco2']
df.columns = cols_py

cols_translation = {}
num_cols = len(cols_orig)
for i in range(num_cols):
    cols_translation[cols_orig[i]] = cols_py[i]

print("Translation for column names:")
# print("\n".join("{}\t{}".format(k, v) for k, v in cols_translation.items()))
for k, v in cols_translation.items():
    print(f"{k:>100} : {v:<50}")

In [ ]:
df.head()

In [ ]:
# Overview of missing values

msno.matrix(df, labels=True)

In [ ]:
# Export 'df' to CSV
df.to_csv('../output/df_monthly_electricity_price.csv', sep=',', index=False)

# Monthly generation capacity

## - Import and inspect data

In [ ]:
file_path = '../data/monthly_features/energy_charts_generation_capacity_germany_from2015.csv'

df_orig_g = pd.read_csv(file_path, delimiter=',')

df_orig_g.head()

In [ ]:
df = df_orig_g.copy()
print(df.shape)
df.info()

In [ ]:
info_describe(df, 2)

In [ ]:
print(df.columns)

In [ ]:
# Overview of missing values

msno.matrix(df, labels=True)

In [ ]:
# Export 'df' to CSV
df.to_csv('../output/df_monthly_solar_capacity.csv', sep=',', index=False)

# Monthly weather average

## - Import and inspect data

In [ ]:
file_path = '../data/monthly_features/weather_monthly_de_from2015.csv'

df_orig_w = pd.read_csv(file_path, delimiter=',')

df_orig_w.head()

In [ ]:
df = df_orig_w.copy()
print(df.shape)
df.info()
df.describe().T

In [ ]:
info_describe(df, 2)

In [ ]:
print(df.columns)

In [ ]:
# Overview of missing values

msno.matrix(df, labels=True)

In [ ]:
# Export 'df' to CSV
df.to_csv('../output/df_monthly_weather_average.csv', sep=',', index=False)